In [1]:
import sys
sys.path.insert(0, "/Users/syue99/research/SimuQ/src/")
import simuq
sys.modules.pop("simuq", None)
import simuq
print(simuq.__file__)
sys.path.insert(0, "../")
from observable_program_generator import *
from simuq import QSystem
from simuq import Qubit
from simuq.hamiltonian import productHamiltonian, TIHamiltonian
from simuq.braket import diffQCProvider as BraketProvider
awsp = BraketProvider()

import numpy as np
import sympy as sp

import numpy as np


import time
sys.path.append('/Users/syue99/research/RISC-Q/PulseDSL/src/Test/utility')
from gatedict import gatedict


import matplotlib.pyplot as plt

import numpy as _np
#from labrad.units import WithUnit
def compile_gates(gate_list = None):
    nor=1000
    """Using the direct AWG output to generate a gate sequence with single and two-qubit gates\n
    channel (int): channel number\n
    amplitudes (Volts): the amplitude of MS\n
    repetition (int): the number of times for this sequence to repeat (with a new trigger)\n
    gate_list [[start_time(us),duration(us), gate1_specified], [start_time,duration, gate2_specified], ..]: gate sequence\n
    avaliable gates:\n
    sigma_phi(float in degs): single qubit gate: e.g. sigma_y: sigma_90\n
    sigmatukey_phi(float in degs): single qubit gate with tukey pulseshaping(a=0.3): e.g. sigmatukey_y: sigma_90\n
    phiphi_phi_mu(MHz)_r1_r2: two qubit gate: e.g. xx MS with mu=1.2MHz and equal strength for bsb and rsb: phiphi_0_1.2_1_1\n
    phiphitukey_phi_mu(MHz)_r1_r2_a: two qubit gate with tukey pulseshaping: e.g. yy MS with mu=1.2MHz and a=0.1 tukey: phiphitukey_90_1.2_1_1_0.1\n
    """
    #we specify gates by using the format of (start_time, duration) and for the start time and duration, there can be two different units
    #1. In the units of regular time s.t. WithUnit(n,'us'). BUT note that n must be divisible by 8. Otherwise we will have an error
    #2. In the units of pi time s.t. WithUnit(n,'pi-time'). This is used for single qubit gates and n can be floats
    
    #first we get total time for the sequence: in units of pi time(2.5us)/actual time + n* 4cycle of 125MHz time (8ns) for n number of gates
    #we also check if the time is implemented correctly
    gate_time_nor = 2504/(1000/nor)
    cycle_time_nor = 8/(1000/nor)
    time_nor = 1/(1000/nor)
    gate_list = _np.array(gate_list)
    #print(type(gate_list))
    #we will replace the original input list to an array with normalized time
    

    #we need an offset array to count the offsets of every starttime due to rounding
    #Note offset[i] is the offset before ith layer
    #offset[0] = 0 always
    offset = _np.zeros(len(gate_list)+1)

    for i in range(len(gate_list)):
        print(gate_list[0])
        start_time = gate_list[i][0]['ns']
        duration = gate_list[i][1]['ns']
        # #print(start_time,duration, time_nor)
        # start_time = _np.ceil(start_time['ns']) * time_nor + offset[i]
        # r = _np.round(_np.remainder(start_time, cycle_time_nor))
        # if r < 0.005 or r > cycle_time_nor - 0.005:
        #     r = 0
        #     #print(r)
        # else:
        #     #pass
        #     r = cycle_time_nor - r
        #     #print(r)
        # offset[i] += r
        # start_time += r
        
        # duration = _np.ceil(duration['ns']) * time_nor
        # r = _np.round(_np.remainder(duration, cycle_time_nor))
        # if r < 0.005 or r > cycle_time_nor - 0.005:
        #     r = 0
        #     #print(r)
        # else:
        #     r = cycle_time_nor - r
        #     #pass
        # #    print(r)
        # offset[i] += r
        # offset[i+1] += offset[i]
        gate_list[i][0] = start_time
        gate_list[i][1] = duration
        #print(start_time,duration)
        """
        #Note: THis is a risky move as we add pi-time as a unit in the labrad.units file

        if start_time.units == 'pi-time':
            start_time = start_time['pi-time'] * gate_time_nor
        else:
            start_time = start_time['ns'] * time_nor
        #first check if the divisible things make sense
        r = _np.abs(_np.remainder(start_time, cycle_time_nor))
        if r < 0.005 or r > cycle_time_nor - 0.005:
            pass
        else:
            print(r)
            raise Exception("wrong gate start time")
        gate_list[i][0] = start_time
        
        if duration == 'pi-time':
            duration = duration['pi-time'] * gate_time_nor
        else:
            duration = duration['ns'] * time_nor
        r = _np.abs(_np.remainder(duration, cycle_time_nor))
        if r < 0.005 or r > cycle_time_nor - 0.005:
            pass
        else:
            print(r)
            raise Exception("wrong gate duration time")
        gate_list[i][1] = duration
        """
        
    total_time = (gate_list[-1][0]+gate_list[-1][1])+len(gate_list)*4*cycle_time_nor
    total_time = int(_np.ceil(total_time))


    wf = _np.zeros(total_time)
    #we need a time_counter to make sure there is no overlap
    time_counter = 0
    for gate_index in range(len(gate_list)):
        gate_start,gate_duration,gate_type = gate_list[gate_index]
        gate_duration = int(_np.round(gate_duration))
        #normalize gate time and gate start time
        #print(gate_start+gate_index*4*cycle_time_nor,gate_duration+4*cycle_time_nor)
        gate_start = int(_np.round(gate_start+gate_index*4*cycle_time_nor))
        gate_duration = int(_np.round(gate_duration+4*cycle_time_nor))
        if gate_start >= time_counter:
            time_counter = gate_start+gate_duration
        else:
            
            raise Exception("gate sequence time not specifying correclty")

        #calculate the gate

        pt = _np.linspace(0,gate_duration-1,gate_duration)
        #now we need to make the gate 
        gate = gate_type.split("_")
        #single qubit gate: gatename, phi
        if len(gate)==2:
            gate_phi = int(float(gate[1])/180*_np.pi)
            start_time_phi = 2*_np.pi*gate_list[i][0]*.08/time_nor
            
            #print(gate_phi)
            gate = gate[0]
            wf[gate_start:gate_start+gate_duration] = gatedict.gatedict[gate](gate_phi+start_time_phi,pt)

        else:
            gate = gate[0]
            wf[gate_start:gate_start+gate_duration] = gatedict.gatedict[gate](pt)
         
    
    return wf
    

def ising(N, T, J, h):
    qs = QSystem()
    q = [Qubit(qs) for _ in range(N)]
    H = 0
    for j in range(N):
        for k in range(N):
            print(J[j,k])
            H += J[j, k] * q[j].Z * q[k].Z
    for j in range(N):
        H += h[j] * q[j].X
    print(H)
    qs.add_evolution(H, T)
    return qs

ising(2,1, np.ones((2, 2)),[1,2])

#Should change a numpy array to an array of functions



/Users/syue99/research/SimuQ/src/simuq/__init__.py
1.0
1.0
1.0
1.0
1.0  +  1.0 * Qubit0.Z * Qubit1.Z  +  1.0 * Qubit0.Z * Qubit1.Z  +  1.0  +  1 * Qubit0.X  +  2 * Qubit1.X


Quantum system:
- Sites: Qubit0 Qubit1 
- Sequentially evolves:
    Time = 1,  TIHamiltonian = 2.0  +  2.0 * Qubit0.Z * Qubit1.Z  +  1 * Qubit0.X  +  2 * Qubit1.X

In [39]:
import sympy as sp
x = sp.symbols('x')
theta = sp.symbols('theta')
J0 = sp.sin(2*x + theta)

qs = QSystem()
q = [Qubit(qs) for _ in range(2)]
H = 0
H += J0 *  q[0].Z *  q[1].Z
H += J0 * q[0].X
H += J0 * q[1].X
#H += J0 *  q[0].Z *  q[1].Z

H = H.set_parameterizedHam({"theta": np.pi/4 - 0.1})

# T=1 keeps dynamics in the perturbative regime where the PSR estimator converges well.
# n_sample=200 gives ~3-4% MC error; use more for tighter estimates.
np.random.seed(42)
T_total = 1.0
listH = observable_program_generator(H, T_total, n_sample=100, n_repetition=1, diff_var="x", value=1.0)
print(f"listH: {len(listH)} Hj term(s), {len(listH[0][0])} H_lists each")
#listH

listH: 3 Hj term(s), 200 H_lists each


In [44]:
# Step 1: Wire QuTiPSequentialRunner
import sys
sys.path.insert(0, "/Users/syue99/research/SimuQ/differential_computing/")
from qutip_sequential import QuTiPSequentialRunner

runner = QuTiPSequentialRunner(n_qubits=2)
psi0   = runner.zero_state()
obs    = runner.zz_observable(0, 1)          # measure ZZ on qubits 0,1
expfn  = runner.make_expectation_fn(psi0, obs)

# Quick sanity check: run a single H_list from listH
#listH[j] = [H_tot_list, evaluated_ugrad, n_rep]
#H_tot_list[0] = first H_list (sgn=-1 for first tau)
test_H_list = listH[0][0][0]
val = expfn(test_H_list)
print(f"Sanity check — <ZZ> after test sequence: {val:.6f}")

Sanity check — <ZZ> after test sequence: 0.808659


In [45]:
# Step 2: Compute gradient via combine_gradient_results
from combine_gradient import combine_gradient_results

diff_var_val = 1.0
grad_estimate = combine_gradient_results(listH, expfn, T=T_total)
print(f"Gradient estimate  d<ZZ>/dx  at x={diff_var_val}, T={T_total}:  {grad_estimate:.6f}")

Gradient estimate  d<ZZ>/dx  at x=1.0, T=1.0:  2.959139


In [46]:
# Step 3: Validate against finite differences
import qutip as qp

def f_at_x(x_val, T_eval=T_total):
    """<ZZ> expectation for the base Hamiltonian evaluated at x=x_val."""
    J_expr = sp.sin(2*sp.Symbol('x') + sp.Symbol('theta'))
    qs_fd = QSystem()
    q_fd  = [Qubit(qs_fd) for _ in range(2)]
    H_fd  = J_expr*q_fd[0].Z*q_fd[1].Z + J_expr*q_fd[0].X + J_expr*q_fd[1].X 
    H_eval = H_fd.set_parameterizedHam({"theta": np.pi/4 - 0.1, "x": x_val})
    psi_fd = qp.tensor([qp.basis(2, 0)] * 2)
    result = qp.sesolve(H_eval.to_qutip_qobj(), psi_fd, [0, float(T_eval)])
    return float(qp.expect(obs, result.states[-1]).real)

eps = 1e-4
fd_grad = (f_at_x(diff_var_val + eps) - f_at_x(diff_var_val - eps)) / (2 * eps)
print(f"Finite-difference   d<ZZ>/dx  at x={diff_var_val}:  {fd_grad:.6f}")
print(f"Parameter-shift est d<ZZ>/dx  at x={diff_var_val}:  {grad_estimate:.6f}")
print(f"Relative error: {abs(grad_estimate - fd_grad) / (abs(fd_grad) + 1e-12):.3%}")

Finite-difference   d<ZZ>/dx  at x=1.0:  2.959129
Parameter-shift est d<ZZ>/dx  at x=1.0:  2.959139
Relative error: 0.000%


In [47]:
J_expr = sp.sin(2*sp.Symbol('x') + sp.Symbol('theta'))
qs_fd = QSystem()
q_fd  = [Qubit(qs_fd) for _ in range(2)]
H_fd  = J_expr*q_fd[0].Z*q_fd[1].Z + J_expr*q_fd[0].X + J_expr*q_fd[1].X + J_expr*q_fd[0].Z*q_fd[1].Z
H_eval = H_fd.set_parameterizedHam({"theta": np.pi/4 - 0.1, "x": diff_var_val})
H_eval

0.8810699280503008 * Qubit0.Z * Qubit1.Z  +  0.4405349640251504 * Qubit0.X  +  0.4405349640251504 * Qubit1.X

In [48]:
qs_fd.add_evolution(H_eval, 1)
qs_fd
awsp = BraketProvider()
awsp.compile(qs_fd, 'quera', 'Aquila', 'rydberg2d', tol=1)
atom_pos = awsp.prog[1]
pulses = awsp.prog[2]
#(i,j) i is instruction type, j is type of hamitlonian:e.g. dressing or gate (kind of buried in the pulse shape of rydberg commands)
#TODO: will need a wrapper on the top of DSL, that handles inst from simqu to awg channel
#e.g. ZZ on 1,3 -> AOD awg moves 1 and 3 to gate zone, then gate awg on with ZZ gate
#TODO: in the instr, native means that it can be together with others, derived cannot, will need to check if dressing is native or derived.
#e.g. gate is definately derived as it will need drag atoms to the gate zone
#TODO: need to add unit so that we know what the pos returned actually means

#TODO: Yuxiang suggests to use sympy to handle the time and other variable part of the hamiltonian, the expression in SIMQU only 
#haneldes the solved global variables or local varibale. E.g. in dressing o is a local varaible that can be solved, maybe d can be a varible handle in sympy
#well, of course for now we just focus on the single qubit case. so that J with sigma_i can be differentiable


for pulse,time in pulses[:1]:
    for p in pulse:
        print(p,time)
pulses


((2, 0), <simuq.qmachine.Instruction object at 0x15633ba60>, 0.425616575129152 * Qubit0.X, [0.851233150258304, -1.1021672219163241e-14]) 1.0350261287765496
((3, 0), <simuq.qmachine.Instruction object at 0x156f3f370>, 0.4256165751647768 * Qubit1.X, [0.8512331503295536, 1.5972439870798565e-14]) 1.0350261287765496
((4, 0), <simuq.qmachine.Instruction object at 0x156f7ee90>, Zero, [-6.380905178638915e-07]) 1.0350261287765496
((5, 0), <simuq.qmachine.Instruction object at 0x156e80340>, 0.8512331502322797 * Qubit0.Z * Qubit1.Z, [0.8512331502322797]) 1.0350261287765496


[([((2, 0),
    0.425616575129152 * Qubit0.X,
    [0.851233150258304, -1.1021672219163241e-14]),
   ((3, 0),
    0.4256165751647768 * Qubit1.X,
    [0.8512331503295536, 1.5972439870798565e-14]),
   ((4, 0),
    Zero,
    [-6.380905178638915e-07]),
   ((5, 0),
    0.8512331502322797 * Qubit0.Z * Qubit1.Z,
    [0.8512331502322797])],
  1.0350261287765496)]

In [49]:
import sys
sys.path.append("/Users/syue99/research/RISC-Q/PulseDSL/src/DSL/")
import os
from PulseDSL_py import *
#print(os.path.dirname(PulseDSL_py.__file__))

num_channels = 6
ch, reg = Channels(num_channels)
schedule = Schedule()
set_platform(PulseLib.Rydberg)

x = 2.0

#p1 = Pulse(shape=Shape.Gaussian, amplitude=0.5, duration=5, phase=0.0, beta=0.1)
#p2 = Pulse(shape=Shape.Square, amplitude=0.9, duration=10, phase=0.0)
p = Pulse(shape=Shape.sigmatukey, amplitude=0.8, duration=7, phase=0.0)
#p4 = Pulse(waveform=lambda t: math.exp(-t**2) * cmath.exp(1j * 2 * math.pi * t), duration=10.0)
#p4 = Pulse(shape=Shape.Drag, amplitude=0.6, duration=10, phase=0.0, beta=0.2)



pulse_inst_dictionary = {0:"detuning 0",1:"detuning 1",2:"raman 0",3:"raman 1",4:"global dressing",5:"2qubit gate"}
#function library
#modelling: each channel into a hardware channel, you have a wrapper on the top of this
for pulse,time in pulses:
    length = len(pulse)
    pulse_list = []
    for inst in range(length):
        DSL_pulse = Pulse(shape=Shape.sigmatukey, amplitude=0.8, duration=time*1e3, phase=0.0)
        pulse_list.append(Play(DSL_pulse,ch[pulse[inst][0][0]],is_par=True))
        #print(pulse_inst_dictionary[pulse[inst][0][0]],time)#,parameters)
    #print(pulse_list)
    Parallel(pulse_list)
    #print(length,pulse[0])


schedule.view()

#schedule.output()


[PulseDSL] startTime=0, ch=2, amp=0.8, freq=0, phase=0.0, addr=0, dur=1035.0261287765497
[PulseDSL] startTime=0, ch=3, amp=0.8, freq=0, phase=0.0, addr=0, dur=1035.0261287765497
[PulseDSL] startTime=0, ch=4, amp=0.8, freq=0, phase=0.0, addr=0, dur=1035.0261287765497
[PulseDSL] startTime=0, ch=5, amp=0.8, freq=0, phase=0.0, addr=0, dur=1035.0261287765497

----------
Schedule view:
----------

Schedule for channel 0:
No entries

Schedule for channel 1:
No entries

Schedule for channel 2:
- sigmatukey amp=0.8 dur=1035.0261287765497 phase=0.0 from 0 to 1035.0261287765497

Schedule for channel 3:
- sigmatukey amp=0.8 dur=1035.0261287765497 phase=0.0 from 0 to 1035.0261287765497

Schedule for channel 4:
- sigmatukey amp=0.8 dur=1035.0261287765497 phase=0.0 from 0 to 1035.0261287765497

Schedule for channel 5:
- sigmatukey amp=0.8 dur=1035.0261287765497 phase=0.0 from 0 to 1035.0261287765497

Schedule for decoder 0:
No entries

End of schedule.
----------



---
## TweezerMapper — Interactive walkthrough

The TweezerMapper translates DiffSimuQ H_list branches into hardware schedule op dicts.
Below we build everything from scratch step by step so you can inspect what each stage produces.

**Unit system throughout:** μm (distance) · μs (time) · rad·μs⁻¹ (angular frequency)


In [ ]:
import sys, os
sys.path.insert(0, "/Users/syue99/research/SimuQ/src/")
sys.path.insert(0, "/Users/syue99/research/SimuQ/differential_computing/")

import numpy as np
import sympy as sp
from unittest.mock import MagicMock

from simuq import QSystem, Qubit

from tweezer_mapper import (
    TweezerMapper, TransportLog, C_6,
    classify_instruction,
    _op_aod, _op_play, _op_delay,
)
from aod_channel import make_aod_pulse, encode_positions, AOD_FREQ_MHZ

print(f"C_6 = {C_6:.1f}  rad·μm⁶·μs⁻¹  (= {C_6/(2*np.pi):.0f} MHz·μm⁶ × 2π)")
print(f"R at J=1 rad/μs: R = (C_6/1)^(1/6) = {C_6**(1/6):.2f} μm")

### Step 1 — Atom layout (sol_gvars)

`generate_as()` stores atom positions in `sol_gvars` as a flat list:
```
sol_gvars = [x1, y1,  x2, y2,  ...]
```
Atom 0 is always fixed at the origin (0, 0).  
We use a 3-qubit equilateral triangle at 6 μm spacing — physical for QuEra Aquila.


In [ ]:
# 3 atoms: atom 0 at origin; atoms 1 and 2 from sol_gvars
# Equilateral triangle, side = 6 μm  →  (0,0), (6,0), (3, 3√3≈5.196)
sol_gvars = [6.0, 0.0,   3.0, 5.196]

mapper = TweezerMapper(n_qubits=3, sol_gvars=sol_gvars, boxes=[], ramp_time=10.0)

print("Atom rest positions (μm):")
for i, pos in enumerate(mapper.rest_positions()):
    print(f"  atom {i}: {pos}")

# Side lengths (should all be ~6 μm)
pos = mapper.rest_positions()
import itertools
for i, j in itertools.combinations(range(3), 2):
    d = np.sqrt((pos[i][0]-pos[j][0])**2 + (pos[i][1]-pos[j][1])**2)
    print(f"  dist({i},{j}) = {d:.3f} μm")

### Step 2 — AOD pulse encoding

The AOD channel encodes atom positions as a **scalar amplitude proxy** (max displacement from origin)
on a fixed 100 MHz sine carrier.  `make_aod_pulse` returns a plain dict — no PulseDSL session needed.
`to_pulsedsl()` in `diffQC_provider.py` is the only place that converts these dicts to actual Pulse objects
(multiplying duration × 1000 to go from μs → ns).


In [ ]:
pos = mapper.rest_positions()
pulse = make_aod_pulse(pos, ramp_time=10.0)   # ramp_time in μs

print("AOD pulse dict:")
for k, v in pulse.items():
    print(f"  {k}: {v}")

print()
print(f"amplitude = max displacement from origin = {encode_positions(pos):.3f} μm")
print(f"duration  = {pulse['duration']} μs  →  {int(pulse['duration']*1000)} ns in PulseDSL")

### Step 3 — classify_instruction

`classify_instruction` reads the instruction `.name` string set by `rydberg2d.py` and routes it.
This is what `map_evaluated_H` calls for every box entry to decide dressing vs. ZZ vs. native.

Channel numbering (n=3 qubits):
- lines 0,1,2 → Detuning of site 0,1,2
- lines 3,4,5 → Rabi of site 0,1,2
- line 6 → dressing global
- lines 7+ → c01_zz, c02_zz, c12_zz, ...


In [ ]:
def fake_ins(name, nativeness="native"):
    m = MagicMock(); m.name = name; m.nativeness = nativeness; return m

tests = [
    ("Detuning of site 0",         "native"),
    ("Rabi of site 2",             "native"),
    ("dressing gloabl potential",  "native"),   # typo in rydberg2d.py is intentional
    ("c01_zz",                     "derived"),
    ("c12_zz",                     "derived"),
    ("mystery",                    "native"),
]
for name, nat in tests:
    ins = fake_ins(name, nat)
    print(f"  {name!r:35s} → {classify_instruction(ins)}")

### Step 4 — map_evaluated_H: dressing box

`boxes` comes from `generate_as()`.  Each box entry is:
```python
((line_idx, ins_idx), ins_object, h_eval, ins_lvars)
```
We build a fake box manually so we can see exactly what `map_evaluated_H` emits.

**Case A: dressing box** — AOD confirms all atom positions, ZZ in same box is suppressed.


In [ ]:
# Dressing box: o_coef=0.8 (scale factor from generate_as)
ins_dress = fake_ins("dressing gloabl potential", "native")
ins_zz    = fake_ins("c01_zz", "derived")

box_dress = ([
    ((6, 0), ins_dress, MagicMock(), [0.8]),   # dressing; ins_lvars[0] = o_coef
    ((7, 0), ins_zz,    MagicMock(), [1.2]),   # ZZ J=1.2 rad/μs — will be suppressed
], 2.0)

m_dress = TweezerMapper(n_qubits=3, sol_gvars=sol_gvars, boxes=[box_dress], ramp_time=10.0)
ops = m_dress.map_evaluated_H(duration=2.0, t_cursor=0.0)

print(f"{len(ops)} ops emitted:")
for op in ops:
    print(" ", op)

print()
print(m_dress.log.summary())
print()
print("Note: ZZ is suppressed because dressing is active in the same box.")

**Case B: ZZ-only box** — dressing absent, so CZ transport is used.  
The pair is moved to `R_target = (C_6 / J)^(1/6)` μm, held for `duration`, then returned to rest.


In [ ]:
ins_zz2 = fake_ins("c12_zz", "derived")
box_zz = ([
    ((7, 0), ins_zz2, MagicMock(), [2.0]),   # J = 2.0 rad/μs
], 1.5)

m_zz = TweezerMapper(n_qubits=3, sol_gvars=sol_gvars, boxes=[box_zz], ramp_time=10.0)
ops_zz = m_zz.map_evaluated_H(duration=1.5, t_cursor=5.0)

print(f"{len(ops_zz)} ops emitted:")
for op in ops_zz:
    print(" ", op)

print()
print(m_zz.log.summary())

J = 2.0
R = (C_6 / J) ** (1/6)
rest_12 = np.sqrt((sol_gvars[0]-sol_gvars[2])**2 + (sol_gvars[1]-sol_gvars[3])**2)
print(f"\nPhysics: J={J} rad/μs  →  R_target = {R:.3f} μm  (rest={rest_12:.3f} μm)")
print("→ atoms move CLOSER (R_target < rest distance) for stronger coupling")

### Step 5 — map_hlist: full 3-segment branch

A full H_list from `observable_program_generator` looks like:
```
[[evaluated_H, tau], [Hj, kick_duration], [evaluated_H, T - tau]]
```
All three segments go through `map_evaluated_H`.  The time cursor advances automatically.


In [ ]:
from observable_program_generator import observable_program_generator

x, theta = sp.symbols("x theta")
J0 = sp.sin(2*x + theta)
qs = QSystem(); q = [Qubit(qs) for _ in range(2)]
H = J0*q[0].Z*q[1].Z + J0*q[0].X + J0*q[1].X + J0*q[0].Z*q[1].Z
H_param = H.set_parameterizedHam({"theta": np.pi/4 - 0.1})

np.random.seed(42)
T = 1.0
programs = observable_program_generator(H_param, T, n_sample=3, n_repetition=1,
                                        diff_var="x", value=1.0)

H_list = programs[0][0][0]   # first gradient term, first H_list

print("H_list (3 segments):")
for i, (H_seg, dur) in enumerate(H_list):
    label = ["evaluated_H (before)", "Hj kick         ", "evaluated_H (after)"][i]
    print(f"  [{i}] {label}  dur={dur:.4f} μs")

In [ ]:
# Use a fake 2-qubit ZZ-only box so we can see ops for all 3 segments
sol_2q = [6.0, 0.0]
ins_zz3 = fake_ins("c01_zz", "derived")
box_2q = ([((3, 0), ins_zz3, MagicMock(), [1.0])], 1.0)

m2q = TweezerMapper(n_qubits=2, sol_gvars=sol_2q, boxes=[box_2q], ramp_time=10.0)
ops_full, log_full = m2q.map_hlist(H_list)

print(f"{len(ops_full)} total ops across 3 segments:")
for op in ops_full:
    print(" ", op)

print()
print(log_full.summary())
print()
print(f"Time check: τ={H_list[0][1]:.4f} + kick={H_list[1][1]:.4f} + (T-τ)={H_list[2][1]:.4f} = {sum(s[1] for s in H_list):.4f} μs")

### Step 6 — diffQCProvider.run(backend="hardware")

The provider wraps everything above.  After `compile()`, calling `run(backend="hardware")`
runs `map_hlist` for every branch of every gradient term and stores:

- `prov._branch_ops`     — list of `(branch_op_lists, ugrad, n_rep)` per gradient term  
- `prov._transport_logs` — list of `TransportLog` lists, same shape


In [ ]:
from simuq.braket.diffQC_provider import diffQCProvider

prov = diffQCProvider()
prov.compile(qs, "quera", "Aquila", "rydberg2d", tol=0.01, verbose=0)

prov.run(programs, obs=None, T=T, backend="hardware", verbose=1)

print()
branch_ops, ugrad, n_rep = prov._branch_ops[0]
print(f"Gradient term 0:  ugrad={float(ugrad):.4f},  {len(branch_ops)} branches")

In [ ]:
# Inspect TransportLog for term 0, branch 0
prov.transport_summary(program_idx=0, branch_idx=0)

In [ ]:
# Count all movements across all branches
total_dress = sum(len(log.dressing_moves)
                  for prog_logs in prov._transport_logs for log in prog_logs)
total_cz    = sum(len(log.cz_moves)
                  for prog_logs in prov._transport_logs for log in prog_logs)
print(f"Total dressing moves: {total_dress}")
print(f"Total CZ moves:       {total_cz}")

# Print all CZ moves for term 0 branch 0
log00 = prov._transport_logs[0][0]
if log00.cz_moves:
    print("\nCZ moves in branch 0:")
    for m in log00.cz_moves:
        print(f"  t={m.t_start:.4f} μs  pair={m.pair}  R={m.R_target:.3f} μm  J={m.J:.4f} rad/μs")
else:
    print("\nNo CZ moves — dressing handles ZZ terms in this compiled box.")

### Step 7 — op dicts → PulseDSL (reference)

`to_pulsedsl(ops, channels, aod_ch)` in `diffQC_provider.py` is the bridge to real hardware.

| op dict `"op"` | PulseDSL call | duration conversion |
|---|---|---|
| `"aod"` | `Play(Pulse(Sine, amp, dur), aod_ch)` | μs × 1000 → ns |
| `"play"` | `Play(Pulse(Constant, amp, phase, dur), ch[i])` | μs × 1000 → ns |
| `"delay"` | `Delay(dur, aod_ch)` | μs × 1000 → ns |

You can inspect any op dict from a branch directly:


In [ ]:
# Grab the first branch from the hardware run
branch_ops_0, _, _ = prov._branch_ops[0]
ops_branch0 = branch_ops_0[0]   # ops for the first H_list

print(f"Branch 0 has {len(ops_branch0)} ops:")
for op in ops_branch0:
    dur_ns = int(op["duration"] * 1000)
    if op["op"] == "aod":
        print(f"  AOD  amp={op['amplitude']:.3f} μm  dur={op['duration']} μs → {dur_ns} ns")
    elif op["op"] == "play":
        print(f"  PLAY ch={op['channel']}  amp={op['amplitude']:.4f} rad/μs  dur={op['duration']:.4f} μs → {dur_ns} ns")
    elif op["op"] == "delay":
        print(f"  DELAY  dur={op['duration']:.4f} μs → {dur_ns} ns")